# Clinical Concept Annotation Benchmarking

This notebook demonstrates how to benchmark clinical concept annotation methods using real SNOMED CT data.

## Overview

Annotation benchmarking evaluates how well a method can identify and rank relevant SNOMED CT concepts for clinical text inputs.

### Key Metrics:
- **Precision@K**: Fraction of top-K predicted concepts that are relevant
- **Recall@K**: Fraction of all relevant concepts found in top-K
- **F1@K**: Harmonic mean of Precision and Recall at K
- **MRR (Mean Reciprocal Rank)**: Average of reciprocal ranks of first relevant concept

### Data Format:
```python
{
    'text': str,           # Clinical text to annotate
    'gold_cuis': List[str]  # Ground truth CUIs
}
```

### Key Changes from Original:
1. Real SNOMED CT concepts are loaded from UK RF2 data
2. ~50 actual SNOMED concept CUIs are sampled (not fake C001, C002)
3. Preferred names from terminology are used as text input
4. Ground truth = actual SNOMED concept IDs that exist in the terminology

In [ ]:
# Import required modules
# For real annotation: import the annotator and term lookup
from snomed_methods import ClinicalConceptAnnotator, create_term_lookup_from_directory
from snomed_methods.benchmarking.annotation import (
    evaluate_annotator,
)

## Load Real SNOMED CT Terminology Data

We load the UK Clinical SNOMED RF2 data to get real concept IDs and their preferred names.

In [ ]:
# Detect UK SNOMED directory
import os

default_snomed_path = "/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z"

# Check if the expected UK SNOMED directory exists
if os.path.exists(default_snomed_path):
    snomed_dir = default_snomed_path
    print(f"Found UK SNOMED data at: {snomed_dir}")

    # Create term lookup from real RF2 data
    term_lookup = create_term_lookup_from_directory(snomed_dir)
    print(f"Loaded {len(term_lookup.concept_ids)} SNOMED concepts")

    # List some concept IDs to verify loading worked
    if len(term_lookup.concept_ids) > 0:
        print("\nSample concepts (first 10):")
        for cui in term_lookup.concept_ids[:10]:
            info = term_lookup.getconcept_info(cui)
            if info and "preferred_name" in info:
                print(f"  {cui}: {info['preferred_name']}")

## Generate Real Annotation Dataset

Sample actual SNOMED concepts and use their preferred names as the text input. The ground truth is the actual concept ID.

In [ ]:
import random

# Sample ~50 real SNOMED concepts if term lookup is available
if term_lookup and len(term_lookup.concept_ids) >= 50:
    # Randomly sample 50 concept IDs from actual terminology
    sampled_cuis = random.sample(term_lookup.concept_ids, 50)

    # For each sampled concept, get its preferred name as the text
    dataset = []
    for cui in sampled_cuis:
        info = term_lookup.getconcept_info(cui)
        if info and "preferred_name" in info:
            # Use a template to create more realistic clinical text
            preferred_name = str(info["preferred_name"])

            # Create varied clinical text based on the concept name
            templates = [
                f"Patient presents with {preferred_name}.",
                f"Diagnosis: {preferred_name}",
                f"The patient has symptoms related to {preferred_name}.",
                f"Medical history includes {preferred_name}.",
                f"Condition {preferred_name} observed during examination.",
            ]
            text = random.choice(templates)

            dataset.append(
                {
                    "text": text,
                    "gold_cuis": [str(cui)],  # Ground truth is the actual SNOMED CUI
                    "preferred_name": preferred_name,
                },
            )

    print(f"Generated dataset with {len(dataset)} real SNOMED concepts")
else:
    # Fallback to generator if no real data available
    from snomed_methods.benchmarking.annotation import generate_annotation_dataset

    print("Using fallback synthetic dataset (no real SNOMED data)")
    dataset = generate_annotation_dataset(num_samples=50)
    # Convert string gold_cuis to list for consistency
    for sample in dataset:
        if isinstance(sample["gold_cuis"], str):
            sample["gold_cuis"] = [sample["gold_cuis"]]

print(f"\nDataset size: {len(dataset)}")
print("\nFirst 3 samples:")
for i, sample in enumerate(dataset[:3]):
    print(f"  Sample {i+1}:")
    print(f"    Text: {sample['text'][:60]}...")
    print(f"    Gold CUIs: {sample['gold_cuis']}")

## Configure the Annotator

We create an annotator that uses real SNOMED CT data for concept matching. If UK data is not available, we use a simplified term lookup approach.

In [ ]:
# Check if we can use the full ClinicalConceptAnnotator with UK data
annotator = None

if snomed_dir and os.path.exists(snomed_dir):
    try:
        # Try to create annotator with UK RF2 data
        # Note: This uses term matching only (no embedding model needed for demo)
        annotator = ClinicalConceptAnnotator(
            uk_path=snomed_dir,
            backend="transformers",  # Use lightweight backend for demo
            device="cpu",
        )
        print("ClinicalConceptAnnotator created with UK RF2 data")

        # Test the annotator with one query
        if dataset:
            test_text = dataset[0]["text"]
            result = annotator.annotate(test_text, top_k=10)
            print("\nTest annotation successful!")
            print(f"Input: {test_text[:50]}...")
            if len(result.top_concepts) > 0:
                print("Top predicted concepts:")
                for i, concept in enumerate(result.top_concepts[:3]):
                    print(
                        f"  {i+1}. {concept.concept_id}: {concept.concept_name} (score: {concept.total_score:.3f})",
                    )
            else:
                print(
                    "  Note: No concepts matched in top 10 - this is expected for synthetic text",
                )
    except Exception as e:
        print(f"Could not create full annotator: {e}")
        print("Will use simplified term lookup approach instead.")
        annotator = None

# If ClinicalConceptAnnotator is not available, create a term-based annotator
if annotator is None and term_lookup:
    print("\nUsing SnomedTermLookup as fallback annotator...")

    def term_lookup_annotator_with_result(text: str, top_k: int = 10):
        """Annotator that returns annotation result with evidence."""
        from snomed_methods import AnnotationResult, MatchedConcept

        words = [w.lower() for w in text.split() if len(w) >= 4]
        results = []
        seen = set()

        for word in words:
            matches = term_lookup.find_concepts_by_term(word, top_n=top_k * 2)
            for cui, name in matches:
                if str(cui) not in seen:
                    seen.add(str(cui))
                    score = 0.8 + (len(word) / len(str(name)) * 0.2)
                    concept = MatchedConcept(str(cui), str(name))
                    concept.add_term_score(word, score)
                    concept.total_score = score
                    results.append(concept)

        result = AnnotationResult()
        result.concept_matches = sorted(
            results,
            key=lambda c: c.total_score,
            reverse=True,
        )[:top_k]
        result.text_terms = [(w, 1.0) for w in words[:5]]
        return result

    annotator_func = term_lookup_annotator_with_result
    print("Term lookup annotator configured")

## Create Real Annotation Function

If the ClinicalConceptAnnotator is properly configured with UK RF2 data, use it. Otherwise, use a keyword-based approach that works with available term lookup.

In [ ]:
# Determine which annotator function to use
if annotator:

    def real_annotator_func(text: str):
        """Use ClinicalConceptAnnotator if available."""
        return annotator.annotate(text, top_k=10)

    print("Using ClinicalConceptAnnotator")
elif "term_lookup_annotator_with_result" in globals():
    real_annotator_func = term_lookup_annotator_with_result
    print("Using SnomedTermLookup-based annotator (fallback)")
else:
    # Heavy fallback: use mock that at least demonstrates the pipeline
    def mock_annotation_result(text):
        """Mock annotation result for demo when no real data available."""
        from snomed_methods import AnnotationResult

        result = AnnotationResult()
        # Return some placeholders to keep the pipeline working
        result.concept_matches = []
        return result

    real_annotator_func = mock_annotation_result
    print("Using placeholder annotator (no SNOMED data available)")

## Evaluate the Annotation Method

Run evaluation using different K values to assess performance at different ranking positions.

In [ ]:
# Convert ground truth gold_cuis from strings to sets for evaluation
# The evaluation expects gold_cuis as a list of strings per sample

# Evaluate using the real annotator function
print("Running annotation benchmark...")
results = evaluate_annotator(
    annotator_func=real_annotator_func,
    dataset=dataset,
    k_values=[1, 3, 5, 10],
)

print("\n=== Annotation Benchmarking Results (Real SNOMED Data) ===")
for metric, value in results.items():
    if metric != "num_samples":
        print(f"{metric}: {value:.4f}")

print(f"\nTotal samples evaluated: {results['num_samples']}")

## Load Pre-generated Datasets

Pre-generated datasets allow for reproducible benchmarking across runs.

In [ ]:
from snomed_methods.benchmarking.annotation import load_annotation_datasets

# Load all pre-generated datasets
datasets_dict = load_annotation_datasets()

for name, data in datasets_dict.items():
    print(f"{name}: {len(data)} samples")

# Use medium dataset for evaluation
medium_dataset = datasets_dict["medium"]
print("\nMedium dataset (first 3 samples):")
for i, sample in enumerate(medium_dataset[:3]):
    print(f"  Sample {i+1}: {sample['text'][:60]}... -> {sample['gold_cuis']}")

## Compare Different K Values

Analyze how precision/recall change with different ranking positions.

In [ ]:
# Evaluate with multiple K values to see the trade-off
k_options = [1, 2, 3, 5, 10]

results_k = evaluate_annotator(
    annotator_func=real_annotator_func,
    dataset=dataset[:30],
    k_values=k_options,
)

print("\nPerformance at different K values:")
print(f"{'K':<5} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print("-" * 45)

for k in k_options:
    p = results_k.get(f"precision@{k}", 0)
    r = results_k.get(f"recall@{k}", 0)
    f1 = results_k.get(f"f1@{k}", 0)
    print(f"{k:<5} {p:<12.4f} {r:<12.4f} {f1:<10.4f}")

## Summary and Key Takeaways

The benchmark now:
1. **Uses real SNOMED CT concepts** - Concepts are sampled from actual UK RF2 data
2. **Has valid ground truth** - Gold CUIs exist in the terminology (not fake C001, C002)
3. **Uses preferred names as text** - Text is derived from real SNOMED concept terminology
4. **Works with ClinicalConceptAnnotator** - If UK data path is valid
5. **Has proper fallback** - Uses SnomedTermLookup when annotator not available

### Expected Metrics:
- **Precision@1**: Should be > 0 when the concept match appears in top-1
- **Precision@K @ K=5,10**: Improves as more concepts are considered
- **Recall@K**: Fraction of relevant concepts found (max = 1.0)
- **F1@K**: Harmonic mean of precision and recall
- **MRR**: Mean reciprocal rank of first relevant concept